In [1]:
import sys, json; sys.path.insert(0, '..')
import pandas as pd, numpy as np
from IPython.display import Image, display
from src import preprocessing as pp, feature_engineering as fe, pipeline as P
from src.config import *
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
def show(name, w=800): display(Image(filename=str(FIG / name), width=w))

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 03 Data cleaning
Score is text: numbers (some with thousands-commas), 'Not Available', or CMS comparison labels.

In [2]:
gen, prov = pp.load_raw(); p = pp.clean_provider(prov)
print('Rows with non-numeric score:', p.score_num.isna().sum())
p.score_label.value_counts()

Rows with non-numeric score: 14175


score_label
Not Available                               9886
No Different than the National Rate         2755
No Different than the National Benchmark     747
Better than the National Rate                322
Worse than the National Rate                 301
Better than the National Benchmark           124
Worse than the National Benchmark             40
Name: count, dtype: int64

In [3]:
p.groupby('Footnote', dropna=False).size()  # footnote meanings: CMS dictionary Table 10

Footnote
1        509
2       2753
3        100
5       3155
6       1174
6,7     1992
7        980
NaN    68767
dtype: int64

In [4]:
wide = pp.pivot_provider(p); pp.invalid_value_report(wide)

,measure,name,n_valid,min,max,invalid_values
0,I_015_01_SIR,C. difficile infection SIR,977,0.00,3.382,0
1,I_013_01_OBS_RATE,Falls with major injury (%),1135,0.00,2.440,0
2,I_022_01_ADJ_RATE,"New/worsened pressure ulcers, risk-adj. (%)",1135,0.00,9.460,0
3,I_017_01_PPR_PD_RSRR,Preventable 30-day post-discharge readmission ...,1126,7.04,12.720,0
4,I_018_01_PPR_WI_RSRR,Preventable within-stay readmission (RSRR),1126,2.90,8.080,0
5,I_019_02_DTC_RS_RATE,"Discharge to community, risk-std. (%)",1126,37.40,82.360,0
6,I_026_01_OBS_RATE,Discharge function score met/exceeded (%),1134,0.00,96.300,0
7,I_021_01_OBS_RATE,Drug regimen review with follow-up (%),1135,0.00,100.000,0
8,I_024_01_OBS_RATE,Med list to next care setting (%),1126,1.17,100.000,0
9,I_025_02_OBS_RATE,Med list to patient/caregiver (%),1027,0.00,100.000,0


Outliers are **reported, not removed** - extreme rates/ratios can be genuine facility observations.

In [5]:
m = pd.read_csv(PROC/'irf_facilities_merged.csv'); pp.outlier_report(m, list(INDEX_MEASURES))

,measure,n,lower_fence,upper_fence,n_outliers
0,I_015_01_SIR,977,-0.77550,1.29250,36
1,I_013_01_OBS_RATE,1135,-0.33750,0.56250,51
2,I_022_01_ADJ_RATE,1135,-1.38000,2.86000,57
3,I_017_01_PPR_PD_RSRR,1126,7.21000,11.69000,12
4,I_018_01_PPR_WI_RSRR,1126,2.77500,6.65500,29
5,I_019_02_DTC_RS_RATE,1126,55.67000,79.49000,34
6,I_026_01_OBS_RATE,1134,9.18125,123.23125,4
7,I_021_01_OBS_RATE,1135,95.82500,102.50500,112
8,I_024_01_OBS_RATE,1126,96.76875,101.93875,175
9,I_025_02_OBS_RATE,1027,95.00000,103.00000,88
